# Progetto Clingo IALAB 2025/2026

Alessio Camerano - Matricola 1044444

## In cosa consiste

Il progetto modella un torneo a turni con partecipanti iniziali e partecipanti che entrano in seguito.
Il sistema genera gli incontri, assegna eventuali BYE, calcola le vittorie e costruisce la classifica finale.

## Come e' stato fatto

- I fatti sui partecipanti sono in `source/clingo/partecipanti.cl`.
- Le regole della classifica sono in `source/clingo/classifica.cl`.
- La gestione dei turni, degli incontri e dei BYE e' in `source/clingo/calendario.cl`.
- I controlli di coerenza sono in `source/clingo/test.cl`.
- Lo script `utility/run_torneo.py` esegue Clingo, legge il modello ottimo e stampa un riepilogo leggibile.

## Risultati

Il modello produce per ogni turno gli incontri, i BYE assegnati e la classifica progressiva.
I test incorporati verificano che ogni partecipante presente compaia nella classifica, che ogni incontro generi un `match/4` coerente e che il vincitore sia uno dei due giocatori.
L'ottimizzazione minimizza la differenza di punti tra gli avversari per rendere gli abbinamenti piu' equilibrati.

In [2]:
from pathlib import Path
import subprocess
import sys

project_root = Path.cwd()
if not (project_root / "utility" / "run_torneo.py").exists():
    project_root = project_root.parent

script_path = project_root / "utility" / "run_torneo.py"
result = subprocess.run([sys.executable, str(script_path)], capture_output=True, text=True, check=True)
print(result.stdout)

--- Turno 1 ---
anna vs maria con anna vincitore
giovanni vs giuseppe con giuseppe vincitore
BYE assegnato a mario
Classifica al turno 1:
1) anna - 1 punto - BYE: no
1) giuseppe - 1 punto - BYE: no
1) mario - 1 punto - BYE: si
4) giovanni - 0 punti - BYE: no
4) maria - 0 punti - BYE: no

--- Turno 2 ---
anna vs giovanni con giovanni vincitore
maria vs mario con mario vincitore
BYE assegnato a giuseppe
Classifica al turno 2:
1) giuseppe - 2 punti - BYE: si
1) mario - 2 punti - BYE: si
3) anna - 1 punto - BYE: no
3) giovanni - 1 punto - BYE: no
5) maria - 0 punti - BYE: no

--- Turno 3 ---
anna vs giuseppe con anna vincitore
giovanni vs mario con mario vincitore
luca vs maria con maria vincitore
BYE assegnato a nessuno
Classifica al turno 3:
1) mario - 3 punti - BYE: si
2) anna - 2 punti - BYE: no
2) giuseppe - 2 punti - BYE: si
4) giovanni - 1 punto - BYE: no
4) maria - 1 punto - BYE: no
6) luca - 0 punti - BYE: no

--- Turno 4 ---
anna vs mario con mario vincitore
giovanni vs maria con

## Nota sul file Python

Il file `utility/run_torneo.py` e' stato generato con Copilot per darmi supporto nel testing e nella presentazione dei risultati.
Non era parte della richiesta iniziale del progetto, ma e' stato aggiunto come strumento di verifica e di lettura dell'output.

## Esempi estratti dai sorgenti Clingo

Di seguito sono mostrati estratti dei file in `source/clingo/` che evidenziano i punti focali dell'implementazione: partecipanti, gestione dei turni e incontri, BYE, calcolo della classifica e controlli di coerenza.

### Partecipanti (source/clingo/partecipanti.cl)

Fatti base e ingressi successivi:

```clingo
% Fatti base
partecipante(mario).
partecipante(giovanni).
partecipante(giuseppe).
partecipante(anna).
partecipante(maria).

% Ingressi successivi
ongoing_partecipante(luca,3).
ongoing_partecipante(elena,5).
```

### Turni, attivi e incontri (source/clingo/calendario.cl)

Regole che definiscono i turni, la partecipazione e la scelta degli incontri:

```clingo
% Turni
turno(1..6).

% Partecipanti attivi
attivo(P,T) :- partecipante(P), turno(T).
attivo(P,T) :- ongoing_partecipante(P,TIngresso), turno(T), T >= TIngresso.

% Coppie candidate
coppia(A,B,T) :- attivo(A,T), attivo(B,T), A < B.

% Incontri scelti
{ incontro(A,B,T) : coppia(A,B,T) }.
```

### Gestione BYE e vincite (source/clingo/calendario.cl)

Snippet che mostra come viene gestito il BYE e l'assegnazione della vittoria:

```clingo
% Gestione BYE
:- attivo(P,T), not bye_assegnato(P,T), not partecipante_in_incontro(P,T).
:- attivo(P,T), bye_assegnato(P,T), partecipante_in_incontro(P,T).
{ bye_assegnato(P,T) } :- attivo(P,T).

vittoria(P,T) :- bye_assegnato(P,T).

% Un solo incontro per coppia
:- incontro(A,B,T1), incontro(A,B,T2), T1 < T2.

1 { vittoria(A,T); vittoria(B,T) } 1 :- incontro(A,B,T).
```

### Calcolo della classifica (source/clingo/classifica.cl)

Come vengono calcolati i punti totali e la posizione finale:

```clingo
% Punteggio e posizione
punti_totali(P,Tot) :-
\tpresente(P),
\tTot = #count { T : vittoria(P,T) }.

posizione(P,Pos) :-
\tpresente(P),
\tpunti_totali(P,Punti),
\tMigliori = #count { Q : presente(Q), punti_totali(Q,AltriPunti), AltriPunti > Punti },
\tPos = 1 + Migliori.
```

### Controlli e output (source/clingo/test.cl)

I test di coerenza inclusi e le direttive `#show` che il runner usa per l'output:

```clingo
% Controlli di coerenza
test_fallito(classifica_incompleta) :- presente(P), not classifica(P,_,_,_).
test_fallito(incontro_senza_match) :- incontro(A,B,T), not match(A,B,_,T).
test_fallito(vincitore_non_coerente) :- match(A,B,V,_), V != A, V != B.
test_superato :- not test_fallito(_).

#show match/4.
#show bye_assegnato/2.
#show vittoria/2.
#show test_superato/0.
#show test_fallito/1.
```

## Come eseguire

```bash
python utility/run_torneo.py
```

Lo script esegue Clingo sui file in `source/clingo/` e stampa il modello risultante con le direttive `#show`.